In [1]:
import os
import sys

import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from semantic_ai_agent.cache import (
    CacheEvaluator,
    CacheResult,
    CacheResults,
    PerfEval,
    SemanticCacheWrapper,
    config,
    load_openai_key,
    try_connect_to_redis,
)

## 1. Environment & Redis Connection

Load environment variables and verify that Redis is reachable.

Make sure Redis is running:
```bash
docker compose up -d
```

In [2]:
load_dotenv()
load_openai_key()

print("Config:", config)
r = try_connect_to_redis(config["redis_url"])

> OpenAI API key is already loaded in the environment
Config: {'redis_url': 'redis://localhost:6379', 'cache_name': 'semantic-cache', 'distance_threshold': 0.3, 'ttl_seconds': 3600}
Redis is running and accessible!


## 2. Initialize the Semantic Cache

Create a `SemanticCacheWrapper` from the project config.

In [3]:
cache = SemanticCacheWrapper.from_config(config)
print(f"Cache '{config['cache_name']}' ready (threshold={config['distance_threshold']})")

Redis is running and accessible!
23:20:13 sentence_transformers.SentenceTransformer INFO   Use pytorch device_name: cpu
23:20:13 sentence_transformers.SentenceTransformer INFO   Load pretrained SentenceTransformer: redis/langcache-embed-v1


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Cache 'semantic-cache' ready (threshold=0.3)


## 3. Hydrate the Cache

Populate the cache with sample question-answer pairs.
Replace this with your own domain data.

In [4]:
sample_data = pd.DataFrame(
    {
        "question": [
            "What is your return policy?",
            "How do I track my order?",
            "Do you offer international shipping?",
            "What payment methods do you accept?",
            "How can I contact customer support?",
        ],
        "answer": [
            "You can return any item within 30 days of purchase for a full refund.",
            "You can track your order using the tracking link sent to your email after shipping.",
            "Yes, we ship to over 50 countries worldwide. Shipping rates vary by destination.",
            "We accept credit cards (Visa, MasterCard, Amex), PayPal, and Apple Pay.",
            "You can reach us via email at support@example.com or call 1-800-555-0123.",
        ],
    }
)

cache.hydrate_from_df(sample_data, q_col="question", a_col="answer")
print(f"Cache hydrated with {len(sample_data)} entries")
sample_data

Cache hydrated with 5 entries


,question,answer
0,What is your return policy?,You can return any item within 30 days of purc...
1,How do I track my order?,You can track your order using the tracking li...
2,Do you offer international shipping?,"Yes, we ship to over 50 countries worldwide. S..."
3,What payment methods do you accept?,"We accept credit cards (Visa, MasterCard, Amex..."
4,How can I contact customer support?,You can reach us via email at support@example....


## 4. Query the Cache

Test with semantically similar queries that don't match the exact cached text.

In [5]:
result = cache.check("Can I get a refund?")
print(f"Query: '{result.query}'")
for match in result.matches:
    print(f"  Match: '{match.prompt}'")
    print(f"  Response: '{match.response}'")
    print(f"  Distance: {match.vector_distance:.4f}")
    print(f"  Cosine Similarity: {match.cosine_similarity:.4f}")

Query: 'Can I get a refund?'


In [6]:
test_queries = [
    "How do I return a product?",
    "Where is my package?",
    "Do you ship outside the US?",
    "What credit cards can I use?",
    "How do I get in touch with you?",
    "What is the meaning of life?",
]

results = cache.check_many(test_queries, show_progress=True)

for r in results:
    hit = "HIT" if r.matches else "MISS"
    distance = f"{r.matches[0].vector_distance:.4f}" if r.matches else "N/A"
    print(f"[{hit}] '{r.query}' -> distance={distance}")

  0%|          | 0/6 [00:00<?, ?it/s]

[MISS] 'How do I return a product?' -> distance=N/A
[MISS] 'Where is my package?' -> distance=N/A
[HIT] 'Do you ship outside the US?' -> distance=0.2023
[MISS] 'What credit cards can I use?' -> distance=N/A
[HIT] 'How do I get in touch with you?' -> distance=0.2420
[MISS] 'What is the meaning of life?' -> distance=N/A


## 5. Evaluate Cache Effectiveness

Use `CacheEvaluator` to measure precision, recall, and F1.

In [7]:
true_labels = [True, True, True, True, True, False]

evaluator = CacheEvaluator(true_labels, results)
metrics = evaluator.get_metrics(distance_threshold=config["distance_threshold"])

print(f"Cache Hit Rate: {metrics['cache_hit_rate']:.2%}")
print(f"Precision:      {metrics['precision']:.2%}")
print(f"Recall:         {metrics['recall']:.2%}")
print(f"F1 Score:       {metrics['f1_score']:.2%}")
print(f"Accuracy:       {metrics['accuracy']:.2%}")

Cache Hit Rate: 33.33%
Precision:      100.00%
Recall:         66.67%
F1 Score:       80.00%
Accuracy:       83.33%


In [8]:
evaluator.matches_df()

,query,match,distance,true_label
0,How do I return a product?,None,NaN,True
1,Where is my package?,None,NaN,True
2,Do you ship outside the US?,Do you offer international shipping?,0.202276,True
3,What credit cards can I use?,None,NaN,True
4,How do I get in touch with you?,How can I contact customer support?,0.242045,True
5,What is the meaning of life?,None,NaN,False


## 6. Performance Tracking

Use `PerfEval` to measure latency across cache lookups.

In [9]:
perf = PerfEval()
perf.set_total_queries(len(test_queries))

with perf:
    for query in test_queries:
        perf.start()
        result = cache.check(query)
        label = "cache_hit" if result.matches else "cache_miss"
        perf.tick(label)

print(perf.summary(labels=["cache_hit", "cache_miss"]))

Performance Summary
Total Queries: 6
Average Latency: 2.6ms
P50 Latency: 3.0ms
P95 Latency: 3.9ms

By Label:
  cache_hit: 2 calls, 3.5ms avg
  cache_miss: 4 calls, 2.2ms avg


## 7. Threshold Sweep

Find the optimal distance threshold by sweeping across a range.

In [10]:
full_results = cache.check_many(test_queries, distance_threshold=1.0, show_progress=True)

full_evaluator = CacheEvaluator.from_full_retrieval(true_labels, full_results)
sweep = full_evaluator.sweep_thresholds(metric_to_maximize="f1_score")

print(f"Best threshold for F1: {sweep['best_threshold']:.3f}")

  0%|          | 0/6 [00:00<?, ?it/s]

Best threshold for F1: 0.404


## 8. Cleanup

Clear the cache when done experimenting.

In [ ]:
# cache.clear()
# print("Cache cleared")